# Clean data and extract images

In [ ]:
import re
import requests
from PIL import Image

import pandas as pd

phitt_import = pd.read_excel("PHITT-spreadsheet-050526.xlsx", index_col=0, sheet_name="FINAL")
phitt_import

In [ ]:
phitt_df = phitt_import[['COUNTRY', 'SUB-REGION', 'REGION', 'DATE OF FIRST STAMP', 'DATE OF FIRST HISTORICAL STAMP', 'TOPIC', 'THEME', 'URL', 'Scott Catalogue Number', 'Stanley Gibbons Catalogue Number', 'Image Source']].copy()
phitt_df = phitt_df.rename(columns={'SUB-REGION': 'subregion'})
phitt_df.columns = phitt_df.columns.str.lower().str.replace(' ', '_')

phitt_df

In [ ]:
stamp_list = list(zip(phitt_df['country'], phitt_df['image_source']))

In [ ]:
stamp_id = 1
for country, url in stamp_list:
    filepath = f"stamp-images/{stamp_id}-{re.sub(r'[^a-z0-9]', '', country.lower())}.jpeg" 

    r = requests.get(url)
    if r.status_code == 200:
        with open(filepath, 'wb') as f:
            f.write(r.content)
    else:
        # for manual download aka blocked by cloudflare
        print(filepath)
        print(url)
        print("---")

    stamp_id += 1

# Add permalink for images

In [ ]:
stamp_id = 1

img_names = []
perm_urls = []
for country, url in stamp_list:
    perm_link = f"https://raw.githubusercontent.com/ChantalMB/PHITT-data/refs/heads/main/stamp-images/{stamp_id}-{re.sub(r'[^a-z0-9]', '', country.lower())}.jpeg"
    img_name = f"{stamp_id}-{re.sub(r'[^a-z0-9]', '', country.lower())}.jpeg"
    img_names.append(img_name)
    perm_urls.append(perm_link)
    stamp_id += 1

perm_urls

In [ ]:
phitt_df["image_name"] = img_names
phitt_df["image_permalink"] = perm_urls

phitt_df

In [ ]:
phitt_df.to_csv('phitt-datasette-prep.csv', index=True)